# Análisis de Bancarización de Mujeres Migrantes en Colombia
## Gran Encuesta Integrada de Hogares (GEIH) 2025 — DANE

**Objetivo:** Construir una base de datos consolidada a partir de tres módulos de la GEIH 2025 para analizar el acceso a productos financieros formales de mujeres migrantes en Colombia.

**Módulos utilizados:**
- **F1** — Características generales, seguridad social en salud y educación (83 variables)
- **F6** — Migración (45 variables)
- **F7** — Datos del hogar y la vivienda (54 variables)

---

## Índice de contenidos

1. [Configuración general e importación de librerías](#seccion-1)
2. [Carga y exploración inicial de módulos](#seccion-2)
3. [Validación de llaves de identificación](#seccion-3)
4. [Limpieza inicial y selección de variables](#seccion-4)
5. [Cruce y consolidación de módulos](#seccion-5)
6. [Análisis descriptivo inicial](#seccion-6)
7. [Identificación de la población de interés](#seccion-7)
8. [Validación y control de calidad](#seccion-8)
9. [Exportación de la base consolidada](#seccion-9)

---

<a id='seccion-1'></a>
## Sección 1 — Configuración general e importación de librerías

Se importan las librerías estándar para análisis de datos y visualización, y se define la configuración global del notebook (rutas de archivos, opciones de pandas y estilo de gráficas).

In [9]:
# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ─── Configuración de pandas ──────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ─── Estilo de visualizaciones ────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

# ─── Rutas de archivos ────────────────────────────────────────────────────────
# Ajusta BASE_DIR si ejecutas el notebook desde una ubicación distinta
BASE_DIR = Path('datos/GEIH_2025')

RUTA_F1 = BASE_DIR / 'F1_Caracteristicas_Generales' / 'F1.csv'
RUTA_F6 = BASE_DIR / 'F6_Migracion'               / 'F6.csv'
RUTA_F7 = BASE_DIR / 'F7_Datos_Hogar_Vivienda'    / 'F7.csv'

RUTA_SALIDA_CONSOLIDADA  = BASE_DIR / 'base_consolidada_GEIH_2025.csv'
RUTA_SALIDA_MIGRANTES    = BASE_DIR / 'base_mujeres_migrantes_GEIH_2025.csv'
RUTA_DICCIONARIO         = BASE_DIR / 'diccionario_variables.json'

print('Rutas configuradas:')
for ruta in [RUTA_F1, RUTA_F6, RUTA_F7]:
    existe = '✓' if ruta.exists() else '✗ NO ENCONTRADO'
    print(f'  {existe}  {ruta}')

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

<a id='seccion-2'></a>
## Sección 2 — Carga y exploración inicial

Cada módulo de la GEIH se carga de manera independiente. Se revisa su estructura (dimensiones, tipos de datos, primeras filas y listado de columnas) antes de cualquier transformación, lo que permite detectar tempranamente problemas de codificación, columnas inesperadas o tipos de datos incorrectos.

In [ ]:
def cargar_modulo(ruta: Path, nombre: str) -> pd.DataFrame:
    """
    Carga un módulo CSV de la GEIH con manejo de errores de codificación.
    
    La GEIH suele venir en latin-1 (ISO-8859-1); si falla, intenta UTF-8.
    El separador predeterminado de DANE es punto y coma.
    """
    for encoding in ('latin-1', 'utf-8'):
        try:
            df = pd.read_csv(ruta, sep=';', encoding=encoding, low_memory=False)
            print(f'[{nombre}] Cargado con encoding={encoding}')
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f'No se pudo leer {ruta} con latin-1 ni UTF-8')


f1 = cargar_modulo(RUTA_F1, 'F1')
f6 = cargar_modulo(RUTA_F6, 'F6')
f7 = cargar_modulo(RUTA_F7, 'F7')

In [ ]:
def explorar_modulo(df: pd.DataFrame, nombre: str) -> None:
    """Imprime un resumen compacto de dimensiones, tipos de datos y primeras filas."""
    print(f'\n{'═'*60}')
    print(f'  MÓDULO {nombre}')
    print(f'{'═'*60}')
    print(f'Dimensiones : {df.shape[0]:,} filas × {df.shape[1]} columnas')
    print(f'\nTipos de datos:')
    print(df.dtypes.value_counts().to_string())
    print(f'\nPrimeras 3 filas:')
    display(df.head(3))
    print(f'\nColumnas ({df.shape[1]}):')
    print(list(df.columns))


for df_mod, nombre_mod in [(f1, 'F1'), (f6, 'F6'), (f7, 'F7')]:
    explorar_modulo(df_mod, nombre_mod)

<a id='seccion-3'></a>
## Sección 3 — Validación de llaves de identificación

La GEIH identifica a cada persona mediante tres campos que conforman una llave compuesta:
- **DIRECTORIO**: identifica el hogar dentro del operativo de campo.
- **SECUENCIA_P**: identifica al hogar dentro del directorio.
- **ORDEN**: identifica a la persona dentro del hogar.

F7 es un módulo a nivel de hogar, por lo que puede usar **HOGAR** en lugar de ORDEN; se documenta esta diferencia antes de cruzar.

Una llave duplicada indicaría un problema en el proceso de recolección o en la descarga de los archivos.

In [ ]:
LLAVES_PERSONA = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']
LLAVES_HOGAR   = ['DIRECTORIO', 'SECUENCIA_P']


def validar_llaves(df: pd.DataFrame, nombre: str, llaves: list) -> None:
    """
    Verifica existencia, nulidad y unicidad de las llaves de identificación.
    Informa sobre duplicados si los hay, sin eliminarlos automáticamente.
    """
    print(f'\n── Módulo {nombre} ──────────────────────────────────')

    # Verificar que las columnas existen
    faltantes = [c for c in llaves if c not in df.columns]
    if faltantes:
        print(f'  ⚠ Columnas NO encontradas: {faltantes}')
        print(f'  Columnas disponibles: {list(df.columns[:20])} ...')
        return
    print(f'  ✓ Columnas {llaves} encontradas')

    # Verificar nulos en las llaves
    nulos = df[llaves].isnull().sum()
    if nulos.any():
        print(f'  ⚠ Valores nulos en llaves:\n{nulos[nulos > 0]}')
    else:
        print(f'  ✓ Sin valores nulos en llaves')

    # Crear llave compuesta y verificar unicidad
    df['_llave'] = df[llaves].astype(str).agg('-'.join, axis=1)
    n_total = len(df)
    n_unicos = df['_llave'].nunique()
    n_dup = n_total - n_unicos

    if n_dup == 0:
        print(f'  ✓ Llave compuesta única ({n_unicos:,} registros únicos)')
    else:
        print(f'  ⚠ {n_dup:,} registros duplicados sobre {n_total:,} totales')
        print(df[df.duplicated('_llave', keep=False)][llaves].head(10))

    df.drop(columns=['_llave'], inplace=True)


# F1 y F6 son a nivel persona → usan DIRECTORIO + SECUENCIA_P + ORDEN
validar_llaves(f1, 'F1', LLAVES_PERSONA)
validar_llaves(f6, 'F6', LLAVES_PERSONA)

# F7 es a nivel hogar → puede usar HOGAR en vez de ORDEN
if 'ORDEN' in f7.columns:
    print('\n── Módulo F7 ──────────────────────────────────')
    print('  ℹ F7 contiene ORDEN: es posible que tenga nivel persona.')
    print('    Se tratará como nivel hogar usando DIRECTORIO + SECUENCIA_P.')
    validar_llaves(f7, 'F7', LLAVES_PERSONA)
elif 'HOGAR' in f7.columns:
    print('\n── Módulo F7 ──────────────────────────────────')
    print('  ℹ F7 usa HOGAR en lugar de ORDEN: nivel de hogar confirmado.')
    print('    El cruce con F1 (nivel persona) asignará los valores del hogar')
    print('    a cada persona del mismo hogar (relación 1:N).')
    validar_llaves(f7, 'F7', LLAVES_HOGAR)
else:
    print('\n  ⚠ F7 no contiene ORDEN ni HOGAR. Revisar estructura del módulo.')

<a id='seccion-4'></a>
## Sección 4 — Limpieza inicial y selección de variables

De los 83/45/54 campos originales se conservan únicamente las variables relevantes para el análisis de bancarización, renombrándolas a etiquetas semánticas en español. Esto reduce la carga de memoria y facilita la lectura del código en etapas posteriores.

> **Nota metodológica:** El renombrado no implica recodificación de valores; las categorías originales del DANE se mantienen intactas para poder consultar el cuestionario oficial cuando sea necesario.

In [ ]:
# ─── Diccionario de variables por módulo ─────────────────────────────────────
# Estructura: {nombre_original: nombre_nuevo}

VARS_F1 = {
    # Llaves de identificación
    'DIRECTORIO' : 'DIRECTORIO',
    'SECUENCIA_P': 'SECUENCIA_P',
    'ORDEN'      : 'ORDEN',
    # Variables sociodemográficas
    'P3271'  : 'sexo_al_nacer',         # 1=Hombre, 2=Mujer
    'P6040'  : 'edad',
    'P6070'  : 'estado_civil',
    'P6080'  : 'autorreconocimiento_etnico',
    'P3042'  : 'nivel_educativo',
    'P6090'  : 'afiliacion_salud',
    'DPTO'   : 'departamento',
    'AREA'   : 'area_urbana_rural',
    'FEX_C18': 'factor_expansion',
}

VARS_F6 = {
    'DIRECTORIO'  : 'DIRECTORIO',
    'SECUENCIA_P' : 'SECUENCIA_P',
    'ORDEN'       : 'ORDEN',
    'P3373S3'     : 'nacido_exterior',             # 1=Sí, 2=No
    'P3374S1'     : 'pais_origen',
    'P3373S3A1'   : 'anio_llegada_colombia',
    'P3373S3A2'   : 'mes_llegada_colombia',
    'P3375S1'     : 'tiempo_en_colombia_meses',
    'P3376'       : 'vivio_otro_pais_6meses',
}

VARS_F7 = {
    'DIRECTORIO'  : 'DIRECTORIO',
    'SECUENCIA_P' : 'SECUENCIA_P',
    # Productos financieros (nivel hogar → se asignará a cada persona del hogar)
    'P5222S1'     : 'producto_fin_cuenta_ahorro',
    'P5222S2'     : 'producto_fin_cuenta_corriente',
    'P5222S3'     : 'producto_fin_deposito_digital',
    'P5222S5'     : 'producto_fin_credito_formal',
    # Características del hogar
    'P4030S1A1'   : 'estrato_socioeconomico',
    'P5090'       : 'tenencia_vivienda',
    'P6008'       : 'total_personas_hogar',
}

# Si F7 tiene ORDEN también (nivel persona), incluirla en las llaves de selección
if 'ORDEN' in f7.columns:
    VARS_F7['ORDEN'] = 'ORDEN'

print('Diccionarios de variables definidos:')
for nombre, d in [('F1', VARS_F1), ('F6', VARS_F6), ('F7', VARS_F7)]:
    print(f'  {nombre}: {len(d)} variables seleccionadas')

In [ ]:
def seleccionar_y_renombrar(df: pd.DataFrame, mapa: dict, nombre: str) -> pd.DataFrame:
    """
    Filtra columnas según el mapa y las renombra.
    Si alguna columna del mapa no existe en el DataFrame se avisa
    pero no se interrumpe el proceso; las columnas ausentes se omiten.
    """
    disponibles = {orig: nuevo for orig, nuevo in mapa.items() if orig in df.columns}
    ausentes    = [orig for orig in mapa if orig not in df.columns]

    if ausentes:
        print(f'  [{nombre}] ⚠ Variables NO encontradas (se omiten): {ausentes}')

    df_sel = df[list(disponibles.keys())].rename(columns=disponibles).copy()
    print(f'  [{nombre}] {df.shape} → {df_sel.shape} (tras selección)')
    return df_sel


print('Seleccionando y renombrando variables...')
f1_sel = seleccionar_y_renombrar(f1, VARS_F1, 'F1')
f6_sel = seleccionar_y_renombrar(f6, VARS_F6, 'F6')
f7_sel = seleccionar_y_renombrar(f7, VARS_F7, 'F7')

print('\nVista rápida F1 (seleccionada):')
display(f1_sel.head(3))
print('\nVista rápida F6 (seleccionada):')
display(f6_sel.head(3))
print('\nVista rápida F7 (seleccionada):')
display(f7_sel.head(3))

<a id='seccion-5'></a>
## Sección 5 — Cruce y consolidación de módulos

### Estrategia de cruce

Se usa **left join** tomando F1 como tabla base porque:
1. F1 contiene a **todas** las personas encuestadas; es el universo poblacional.
2. F6 (migración) solo tiene registros para personas que respondieron ese módulo; el left join conserva a toda la población y deja NaN donde no hay información de migración.
3. F7 está a **nivel de hogar**: cada fila representa un hogar, no una persona. Al cruzarlo con F1 (nivel persona), los datos del hogar se replican para cada persona del mismo hogar. Esto es metodológicamente correcto pero implica que las variables de F7 reflejan la situación del hogar, no de la persona individualmente.

> **Limitación documentada:** Las variables de productos financieros (P5222) provienen de F7 y miden la tenencia a nivel de hogar. No es posible saber con estos datos qué persona específica del hogar es titular del producto.

In [ ]:
LLAVES_P = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']
LLAVES_H = ['DIRECTORIO', 'SECUENCIA_P']

# Determinar si F7 se cruza por llave persona o llave hogar
llaves_f7 = LLAVES_P if 'ORDEN' in f7_sel.columns else LLAVES_H

# ─── Cruce 1: F1 (base) ← F6 (migración) ────────────────────────────────────
n_antes = len(f1_sel)
base = f1_sel.merge(
    f6_sel,
    how='left',
    on=LLAVES_P,
    suffixes=('', '_f6')
)
n_despues = len(base)

print(f'Cruce F1 ← F6')
print(f'  F1 base            : {n_antes:,} registros')
print(f'  Con información F6 : {base["nacido_exterior"].notna().sum():,} registros')
print(f'  Sin información F6 : {base["nacido_exterior"].isna().sum():,} registros')
print(f'  Registros totales  : {n_despues:,} (debe ser igual a F1)')
assert n_antes == n_despues, 'El cruce generó filas extra — revisar duplicados en F6'

# ─── Cruce 2: base ← F7 (hogar/vivienda) ────────────────────────────────────
n_antes2 = len(base)
base = base.merge(
    f7_sel,
    how='left',
    on=llaves_f7,
    suffixes=('', '_f7')
)
n_despues2 = len(base)

print(f'\nCruce base ← F7')
print(f'  Antes del cruce    : {n_antes2:,} registros')
print(f'  Después del cruce  : {n_despues2:,} registros')
if n_despues2 > n_antes2:
    print(f'  ⚠ El cruce con F7 generó {n_despues2 - n_antes2:,} filas extra.')
    print('    Esto puede ocurrir si F7 tiene múltiples filas por hogar.')
    print('    Se conserva la primera ocurrencia por llave de hogar.')
    # Desduplicar: en un cruce con duplicados en F7, nos quedamos con la primera fila
    base = base.drop_duplicates(subset=LLAVES_P, keep='first')
    print(f'    Tras deduplicar  : {len(base):,} registros')

print(f'\nDimensión final de la base consolidada: {base.shape}')
display(base.head(10))

In [ ]:
# ─── Tabla de cobertura ───────────────────────────────────────────────────────
# Mide qué porcentaje de personas tienen información de cada módulo

cobertura = pd.DataFrame({
    'Fuente': ['F1 (base)', 'F6 — Migración', 'F7 — Hogar/Vivienda', 'F7 — Productos financieros'],
    'Variable_indicador': [
        'sexo_al_nacer',
        'nacido_exterior',
        'tenencia_vivienda',
        'producto_fin_cuenta_ahorro',
    ]
})

cobertura['N_con_info'] = cobertura['Variable_indicador'].apply(
    lambda v: base[v].notna().sum() if v in base.columns else 0
)
cobertura['Pct_cobertura'] = (cobertura['N_con_info'] / len(base) * 100).round(2)

print('Tabla de cobertura de información por módulo:')
display(cobertura[['Fuente', 'N_con_info', 'Pct_cobertura']].rename(
    columns={'N_con_info': 'N con información', 'Pct_cobertura': '% cobertura'}
))

<a id='seccion-6'></a>
## Sección 6 — Análisis descriptivo inicial

Antes de filtrar la población de interés se describe la base consolidada completa para entender la composición general de la muestra y detectar posibles sesgos en la selección.

In [ ]:
# ─── Distribución por sexo ───────────────────────────────────────────────────
etiquetas_sexo = {1: 'Hombre', 2: 'Mujer'}

dist_sexo = (
    base['sexo_al_nacer']
    .map(etiquetas_sexo)
    .value_counts()
    .to_frame('N')
)
dist_sexo['%'] = (dist_sexo['N'] / dist_sexo['N'].sum() * 100).round(2)

print('Distribución por sexo al nacer:')
display(dist_sexo)

In [ ]:
# ─── Distribución por estatus migratorio ─────────────────────────────────────
# nacido_exterior: 1=Sí, 2=No; NaN = no respondió módulo F6

etiquetas_mig = {1: 'Nacido en el exterior', 2: 'Nacido en Colombia'}

dist_mig = (
    base['nacido_exterior']
    .map(etiquetas_mig)
    .fillna('Sin información (no módulo F6)')
    .value_counts()
    .to_frame('N')
)
dist_mig['%'] = (dist_mig['N'] / dist_mig['N'].sum() * 100).round(2)

print('Distribución por estatus migratorio:')
display(dist_mig)

In [ ]:
# ─── Tabla cruzada: sexo × estatus migratorio ────────────────────────────────

base['_sexo_lab']  = base['sexo_al_nacer'].map(etiquetas_sexo)
base['_mig_lab']   = base['nacido_exterior'].map(etiquetas_mig).fillna('Sin info F6')

tabla_cruzada_mig = pd.crosstab(
    base['_sexo_lab'],
    base['_mig_lab'],
    margins=True,
    margins_name='Total'
)
print('Tabla cruzada: Sexo × Estatus migratorio (N personas):')
display(tabla_cruzada_mig)

In [ ]:
# ─── Tabla cruzada: sexo × nivel educativo ───────────────────────────────────

if 'nivel_educativo' in base.columns:
    tabla_educ = pd.crosstab(
        base['_sexo_lab'],
        base['nivel_educativo'],
        margins=True,
        margins_name='Total'
    )
    print('Tabla cruzada: Sexo × Nivel educativo (N personas):')
    display(tabla_educ)
else:
    print('Variable nivel_educativo no disponible en la base.')

In [ ]:
# ─── Análisis de edad por sexo y estatus migratorio ──────────────────────────

if 'edad' in base.columns:
    resumen_edad = (
        base.groupby(['_sexo_lab', '_mig_lab'])['edad']
        .agg(N='count', Media='mean', DE='std', Min='min', Max='max')
        .round(2)
    )
    print('Edad (años) por sexo y estatus migratorio:')
    display(resumen_edad)

In [ ]:
# ─── Tenencia de productos financieros por sexo ───────────────────────────────
# Se codifican las variables dicotómicas (1=Sí) antes de sumar

vars_productos = [
    'producto_fin_cuenta_ahorro',
    'producto_fin_cuenta_corriente',
    'producto_fin_deposito_digital',
    'producto_fin_credito_formal',
]
vars_pres = [v for v in vars_productos if v in base.columns]

if vars_pres:
    # Crear indicador: 1 si tiene AL MENOS un producto financiero formal
    base['tiene_producto_financiero'] = (base[vars_pres] == 1).any(axis=1).astype(int)

    tabla_prod = (
        base.groupby('_sexo_lab')['tiene_producto_financiero']
        .agg(
            N_total='count',
            N_con_producto='sum',
        )
    )
    tabla_prod['Pct_bancarizados'] = (
        tabla_prod['N_con_producto'] / tabla_prod['N_total'] * 100
    ).round(2)

    print('Tenencia de al menos un producto financiero formal por sexo (nivel hogar):')
    display(tabla_prod)

    # Visualización
    fig, ax = plt.subplots()
    tabla_prod['Pct_bancarizados'].plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
    ax.set_title('Porcentaje con producto financiero formal por sexo\n(información a nivel hogar — GEIH 2025)')
    ax.set_ylabel('% del grupo')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    print('\n> Nota: las variables de productos financieros se miden a nivel de hogar en F7.')
    print('> No es posible identificar qué persona del hogar es titular del producto.')
else:
    print('Variables de productos financieros no disponibles.')

<a id='seccion-7'></a>
## Sección 7 — Identificación de la población de interés

La población objetivo son **mujeres nacidas en el exterior** (migrantes internacionales). Se aplican dos filtros secuenciales y se analiza en detalle su perfil sociodemográfico.

Se crea además la variable `es_mujer_migrante` sobre la base completa, útil para análisis comparativos posteriores.

In [ ]:
# ─── Variable binaria en la base completa ─────────────────────────────────────
base['es_mujer_migrante'] = (
    (base['sexo_al_nacer'] == 2) &       # Mujer
    (base['nacido_exterior'] == 1)        # Nacida en el exterior
).astype(int)

print(f'Mujeres migrantes en la base consolidada: {base["es_mujer_migrante"].sum():,}')
print(f'Porcentaje sobre total: {base["es_mujer_migrante"].mean()*100:.2f}%')

# ─── Subconjunto de mujeres migrantes ─────────────────────────────────────────
mujeres_migrantes = base[base['es_mujer_migrante'] == 1].copy()
print(f'\nDimensión del subconjunto: {mujeres_migrantes.shape}')

In [ ]:
# ─── Perfil de mujeres migrantes ─────────────────────────────────────────────

# Distribución por país de origen
if 'pais_origen' in mujeres_migrantes.columns:
    top_paises = mujeres_migrantes['pais_origen'].value_counts().head(15)
    print('Top 15 países de origen de mujeres migrantes:')
    display(top_paises.to_frame('N'))

    fig, ax = plt.subplots(figsize=(10, 5))
    top_paises.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Países de origen más frecuentes — Mujeres migrantes (GEIH 2025)')
    ax.set_xlabel('N personas')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Distribución por departamento de residencia actual ──────────────────────
if 'departamento' in mujeres_migrantes.columns:
    dist_dpto = mujeres_migrantes['departamento'].value_counts().head(20)
    print('Top 20 departamentos de residencia — Mujeres migrantes:')
    display(dist_dpto.to_frame('N'))

In [ ]:
# ─── Distribución por nivel educativo ────────────────────────────────────────
if 'nivel_educativo' in mujeres_migrantes.columns:
    dist_educ = mujeres_migrantes['nivel_educativo'].value_counts().sort_index()
    print('Distribución por nivel educativo — Mujeres migrantes:')
    display(dist_educ.to_frame('N'))

# ─── Estadísticas de edad ─────────────────────────────────────────────────────
if 'edad' in mujeres_migrantes.columns:
    print('\nEstadísticas de edad — Mujeres migrantes:')
    print(mujeres_migrantes['edad'].describe().round(2))

<a id='seccion-8'></a>
## Sección 8 — Validación y control de calidad

Se genera un reporte sistemático de valores faltantes y outliers. Este diagnóstico es esencial antes de cualquier modelado estadístico, ya que la GEIH puede contener missing values por diseño (no aplicación de módulos) o por no respuesta.

In [ ]:
# ─── Reporte de valores faltantes ────────────────────────────────────────────

missing = pd.DataFrame({
    'N_missing'  : base.isnull().sum(),
    'Pct_missing': (base.isnull().mean() * 100).round(2),
}).sort_values('Pct_missing', ascending=False)

missing = missing[missing['N_missing'] > 0]

print(f'Variables con valores faltantes ({len(missing)} de {base.shape[1]}):')
display(missing)

In [ ]:
# ─── Heatmap de missingness ───────────────────────────────────────────────────
# Se grafica sobre una muestra aleatoria para no saturar la visualización

n_muestra = min(500, len(base))
muestra_missing = base.sample(n_muestra, random_state=42).isnull()

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    muestra_missing.T,
    cbar=False,
    cmap='Blues',
    ax=ax,
    xticklabels=False,
    yticklabels=True
)
ax.set_title(
    f'Mapa de missingness (muestra n={n_muestra})\n'
    'Azul = valor faltante — Blanco = valor presente'
)
ax.set_xlabel('Observaciones')
plt.tight_layout()
plt.show()

print('\n> Las variables de F6 (migración) y F7 tienen más missing por diseño:')
print('> F6 solo se aplica a un subconjunto de personas; F7 a nivel de hogar.')

In [ ]:
# ─── Detección de outliers en variables numéricas ────────────────────────────
# Se usa el criterio IQR (Tukey): valores fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]

vars_numericas = ['edad', 'tiempo_en_colombia_meses']
vars_num_pres  = [v for v in vars_numericas if v in base.columns]

print('Detección de outliers (criterio IQR):\n')
for var in vars_num_pres:
    serie = base[var].dropna()
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr   = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    n_outliers = ((serie < lim_inf) | (serie > lim_sup)).sum()

    print(f'  {var}:')
    print(f'    Rango IQR válido : [{lim_inf:.1f}, {lim_sup:.1f}]')
    print(f'    Outliers         : {n_outliers:,} ({n_outliers/len(serie)*100:.2f}%)')
    print(f'    Min / Max        : {serie.min():.1f} / {serie.max():.1f}\n')

In [ ]:
# ─── Consistencia: año de llegada vs tiempo en Colombia ───────────────────────
# Inconsistencia: año_llegada + tiempo_meses/12 debería aproximarse al año actual

ANIO_ACTUAL = 2025

if {'anio_llegada_colombia', 'tiempo_en_colombia_meses'}.issubset(base.columns):
    sub = base[['anio_llegada_colombia', 'tiempo_en_colombia_meses']].dropna()

    # Año implicado por el tiempo en Colombia
    sub['anio_implicado'] = ANIO_ACTUAL - (sub['tiempo_en_colombia_meses'] / 12).round()
    sub['diferencia_anios'] = (sub['anio_llegada_colombia'] - sub['anio_implicado']).abs()

    umbral = 2  # Tolerancia de ±2 años por redondeo en la encuesta
    n_inconsistentes = (sub['diferencia_anios'] > umbral).sum()

    print(f'Validación de consistencia temporal (año llegada vs tiempo en Colombia):')
    print(f'  Registros con dato en ambas variables : {len(sub):,}')
    print(f'  Inconsistentes (diferencia > {umbral} años) : {n_inconsistentes:,}')
    print(f'  Porcentaje inconsistente             : {n_inconsistentes/len(sub)*100:.2f}%')

    if n_inconsistentes > 0:
        print('\n  → Se recomienda revisar estos registros antes del modelado.')
        print('    Una posible causa es el recodigo de tiempo_en_colombia_meses')
        print('    con valores que representan rangos, no meses exactos.')
else:
    print('Variables de consistencia temporal no disponibles.')

<a id='seccion-9'></a>
## Sección 9 — Exportación de la base consolidada

Se exportan tres archivos:
1. **base_consolidada_GEIH_2025.csv** — base completa con todas las personas.
2. **base_mujeres_migrantes_GEIH_2025.csv** — subconjunto de la población de interés.
3. **diccionario_variables.json** — mapa entre nombres originales del DANE y nombres semánticos adoptados en este análisis.

In [ ]:
# ─── Eliminar columnas auxiliares antes de exportar ──────────────────────────
cols_auxiliares = ['_sexo_lab', '_mig_lab']
base_export = base.drop(columns=[c for c in cols_auxiliares if c in base.columns])
mm_export   = mujeres_migrantes.drop(columns=[c for c in cols_auxiliares if c in mujeres_migrantes.columns])

# ─── Guardar bases ───────────────────────────────────────────────────────────
BASE_DIR.mkdir(parents=True, exist_ok=True)

base_export.to_csv(RUTA_SALIDA_CONSOLIDADA, index=False, encoding='utf-8-sig')
mm_export.to_csv(RUTA_SALIDA_MIGRANTES, index=False, encoding='utf-8-sig')

print(f'Base consolidada exportada   : {RUTA_SALIDA_CONSOLIDADA}')
print(f'  {len(base_export):,} filas × {base_export.shape[1]} columnas')
print(f'\nBase mujeres migrantes export: {RUTA_SALIDA_MIGRANTES}')
print(f'  {len(mm_export):,} filas × {mm_export.shape[1]} columnas')

In [ ]:
# ─── Generar diccionario de variables ────────────────────────────────────────

diccionario = {
    'descripcion': 'Diccionario de variables GEIH 2025 — análisis bancarización mujeres migrantes',
    'fecha_generacion': '2025',
    'fuente': 'DANE — Gran Encuesta Integrada de Hogares 2025',
    'variables': {}
}

# Combinar los tres mapas en un solo diccionario
for modulo, mapa in [('F1', VARS_F1), ('F6', VARS_F6), ('F7', VARS_F7)]:
    for original, nuevo in mapa.items():
        if original not in diccionario['variables']:
            diccionario['variables'][original] = {
                'nombre_nuevo'   : nuevo,
                'modulo_origen'  : modulo,
            }

# Variables creadas en el análisis
diccionario['variables']['tiene_producto_financiero'] = {
    'nombre_nuevo' : 'tiene_producto_financiero',
    'modulo_origen': 'derivada',
    'descripcion'  : '1 si el hogar tiene al menos un producto financiero formal (cuenta ahorro, corriente, depósito digital o crédito formal)'
}
diccionario['variables']['es_mujer_migrante'] = {
    'nombre_nuevo' : 'es_mujer_migrante',
    'modulo_origen': 'derivada',
    'descripcion'  : '1 si la persona es mujer (sexo_al_nacer==2) Y nació en el exterior (nacido_exterior==1)'
}

with open(RUTA_DICCIONARIO, 'w', encoding='utf-8') as f:
    json.dump(diccionario, f, ensure_ascii=False, indent=2)

print(f'Diccionario exportado: {RUTA_DICCIONARIO}')
print(f'  Total de variables documentadas: {len(diccionario["variables"])}')

---
## Resumen de resultados

| Resultado | Valor |
|---|---|
| Registros en F1 (universo) | ver celda de carga |
| Registros con información migración (F6) | ver tabla de cobertura |
| Registros con información hogar (F7) | ver tabla de cobertura |
| **Mujeres migrantes identificadas** | ver Sección 7 |

### Limitaciones principales

1. **Nivel de análisis financiero:** Las variables `P5222` (productos financieros) se recolectan a nivel de hogar en F7. No es posible determinar qué integrante del hogar es titular del producto.
2. **Módulo F6 opcional:** No toda persona en F1 respondió el módulo de migración; la variable `nacido_exterior` tiene missing para la mayoría de la muestra, lo cual es esperado por el diseño de la encuesta.
3. **Consistencia temporal:** Se documentaron posibles inconsistencias entre `anio_llegada_colombia` y `tiempo_en_colombia_meses` (ver Sección 8).
4. **Expansión muestral:** El análisis descriptivo no utiliza ponderadores (`factor_expansion`). Para producir estimaciones representativas de la población colombiana se debe multiplicar por `FEX_C18`.

---
*Notebook generado para análisis de bancarización de mujeres migrantes — GEIH 2025, DANE Colombia.*